In [1]:
import os
api_key = os.environ['AZURE_OPENAI_API_KEY']
base_url = os.environ['AZURE_OPENAI_ENDPOINT']
base_url = f"{base_url}/openai/v1"
from openai import OpenAI
client = OpenAI(api_key=api_key, base_url=base_url)

In [2]:
import requests,json
def get_current_weather(city:str)->dict:

    api_key = "6a8b0ac166a37e2b7a38e64416b3c3fe"

    url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}"
    response = requests.get(url)
    response = response.content.decode()
    response = json.loads(response)
    output = {"City Name":city,"weather":response["weather"][0]['description'],
              "temperature":response['main']['temp'],
              "unit":"kelvin"}

    return output

In [3]:
get_current_weather("delhi")

{'City Name': 'delhi',
 'weather': 'scattered clouds',
 'temperature': 310.48,
 'unit': 'kelvin'}

In [4]:
get_current_weather("mumbai")

{'City Name': 'mumbai',
 'weather': 'clear sky',
 'temperature': 303.97,
 'unit': 'kelvin'}

In [5]:
tools = [{"type":"function",
          "name":"get_current_weather",
          "description":"This function can be used to fetch current weather information for any given city or location.",
          "parameters":{"type":"object",
                        "properties":{"city":{"type":"string","description":"name of location or city, e.g. mumbai, new york"},
                                      },
                        },
                        "required":['city'],
},
]

In [14]:
tool_map = {"get_current_weather":get_current_weather}

def generate_resp(prompt,model='gpt-4o-mini'):
    messages = [{"role":"user",'content':prompt}]

    while True:
        response = client.responses.create(model=model,input=messages,
                                           instructions="you are helpful assistant",
                                       tools=tools)
        response_op = response.output
        function_calls = [item for item in response_op if item.type=="function_call"]
        if function_calls:
            print(response_op)
            messages = messages + response_op

            for tool in function_calls:
                tool_name = tool.name
                tool_args = json.loads(tool.arguments)
                tool_func = tool_map[tool_name]
                tool_output = tool_func(**tool_args)
                messages = messages + [{"type":"function_call_output","call_id":tool.call_id,
                                        "output":json.dumps(tool_output)}]
        else:
            break
    return response.output_text
        
        
   

In [15]:
print(generate_resp("what is AI?"))

Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are designed to think and learn like humans. It encompasses various technologies and methodologies that enable computers to perform tasks that traditionally require human intelligence, such as:

1. **Learning**: Acquiring new information and rules for using it.
2. **Reasoning**: Using rules to reach approximate or definite conclusions.
3. **Problem-Solving**: Finding solutions to complex problems.
4. **Perception**: Interpreting sensory data (e.g., visual, auditory).
5. **Natural Language Processing (NLP)**: Understanding and interacting with human language.

AI can be classified into two main types:

- **Narrow AI**: Designed to perform specific tasks (e.g., facial recognition, recommendation systems).
- **General AI**: Hypothetical, capable of performing any intellectual task a human can do.

AI technologies are used in various fields, including healthcare, finance, transportation, and entert

In [16]:
generate_resp("what is the weather today in manali?")

[ResponseFunctionToolCall(arguments='{"city":"Manali"}', call_id='call_RXuKHtpuRVx34qBMimBx8ZrL', name='get_current_weather', type='function_call', id='fc_0ca43ea24867af450069aeab734f488197b1785c5d82e6b356', status='completed')]


'Today in Manali, the weather is clear with a temperature of approximately 300.75 K, which is about 27.6°C.'

In [18]:
print(generate_resp("what is the weather today in manali and delhi?"))

[ResponseFunctionToolCall(arguments='{"city":"Manali"}', call_id='call_XPHdpFRDBb3X26glA3fcj4Vm', name='get_current_weather', type='function_call', id='fc_04b451ab70846ae70069aeabc91dd88193bc1c889843ed5f3d', status='completed'), ResponseFunctionToolCall(arguments='{"city":"Delhi"}', call_id='call_A3DdSza6UXV1EWJRpB9WT01F', name='get_current_weather', type='function_call', id='fc_04b451ab70846ae70069aeabc91de88193b31f0b2a381a991e', status='completed')]
### Today's Weather

**Manali:**
- **Condition:** Clear sky 
- **Temperature:** 27.6°C (300.75 K)

**Delhi:**
- **Condition:** Scattered clouds 
- **Temperature:** 37.3°C (310.48 K)

If you need more details, feel free to ask!


In [20]:
print(generate_resp("What is capital city of India and What is weather there?"))

[ResponseFunctionToolCall(arguments='{"city":"New Delhi"}', call_id='call_UhqF7454hxXBcnhmn0ix2iSU', name='get_current_weather', type='function_call', id='fc_0481da2b9751cc8f0069aeac1ef36481969ae1af08e806d6c5', status='completed')]
The capital city of India is **New Delhi**. Currently, the weather there is characterized by **scattered clouds** with a temperature of approximately **310.37 K** (about 37.2 °C).


In [21]:
import wikipedia

def get_wikipedia_summary(query:str):
    summary = wikipedia.summary(query)
    return summary

In [24]:
get_wikipedia_summary("headquarter city of Google")

'Google LLC ( , GOO-gəl) is an American multinational technology corporation focused on information technology, online advertising, search engine technology, email, cloud computing, software, quantum computing, e-commerce, consumer electronics, and artificial intelligence (AI). It has been referred to as "the most powerful company in the world" by the BBC, and is one of the world\'s most valuable brands. Google\'s parent company Alphabet Inc. has been described as a Big Tech company. \nGoogle was founded on September 4, 1998, by American computer scientists Larry Page and Sergey Brin. Together, they own about 14% of its publicly listed shares and control 56% of its stockholder voting power through super-voting stock. The company went public via an initial public offering (IPO) in 2004. In 2015, Google was reorganized as a wholly owned subsidiary of Alphabet Inc. Google is Alphabet\'s largest subsidiary and is a holding company for Alphabet\'s internet properties and interests. Sundar P

In [25]:
tools = [{"type":"function",
          "name":"get_current_weather",
          "description":"This function can be used to fetch current weather information for any given city or location.",
          "parameters":{"type":"object",
                        "properties":{"city":{"type":"string","description":"name of location or city, e.g. mumbai, new york"},
                                      },
                        },
                        "required":['city'],
},
{"type":"function",
          "name":"get_wikipedia_summary",
          "description":"This function can be used to get wikipedia summary for any given tpopic, place, events or people.",
          "parameters":{"type":"object",
                        "properties":{"query":{"type":"string","description":"name of location , people, events"},
                                      },
                        },
                        "required":['query'],
},
]

In [26]:
tool_map = {"get_current_weather":get_current_weather,
            "get_wikipedia_summary":get_wikipedia_summary}

def generate_resp(prompt,model='gpt-4o-mini'):
    messages = [{"role":"user",'content':prompt}]

    while True:
        response = client.responses.create(model=model,input=messages,
                                           instructions="you are helpful assistant",
                                       tools=tools)
        response_op = response.output
        function_calls = [item for item in response_op if item.type=="function_call"]
        if function_calls:
            print(response_op)
            messages = messages + response_op

            for tool in function_calls:
                tool_name = tool.name
                tool_args = json.loads(tool.arguments)
                tool_func = tool_map[tool_name]
                tool_output = tool_func(**tool_args)
                messages = messages + [{"type":"function_call_output","call_id":tool.call_id,
                                        "output":json.dumps(tool_output)}]
        else:
            break
    return response.output_text
        
        
   

In [27]:
print(generate_resp("tell me more about history of london and its current weather"))

[ResponseFunctionToolCall(arguments='{"query":"History of London"}', call_id='call_zod0En9aGmWl1811RAoxkBOV', name='get_wikipedia_summary', type='function_call', id='fc_006f3fd0c310139b0069aeb2d629cc8196ada4214d6e367dd3', status='completed'), ResponseFunctionToolCall(arguments='{"city":"London"}', call_id='call_oy4kpdTsNiL1pvW03jP1a19G', name='get_current_weather', type='function_call', id='fc_006f3fd0c310139b0069aeb2d629dc819683f5bdc3c199797e', status='completed')]
### History of London
The history of London spans over **2000 years**, making it a significant and vibrant metropolis. Originally founded by the Romans as **Londinium**, it has grown to become one of the world's leading financial and cultural hubs. Throughout its history, London has endured challenges such as:

- **Plagues**: The Black Death reduced the population drastically.
- **Fires**: The Great Fire of 1666 destroyed much of the city.
- **Civil War**: Internal conflicts have shaped its political landscape.
- **Aerial B

In [29]:
print(generate_resp("IN which city world war II started, tell me more about the current weather there. Also tell me weather for Madrid"))

[ResponseFunctionToolCall(arguments='{"query":"World War II"}', call_id='call_IVduaGwvFl9fDs8NkSLgFd5o', name='get_wikipedia_summary', type='function_call', id='fc_0f3512ecc7f4f5b10069aeb3a7f7748196910d28ce79ff7e34', status='completed'), ResponseFunctionToolCall(arguments='{"city":"Madrid"}', call_id='call_iaYksIxOcbTK074CVcAwMq9B', name='get_current_weather', type='function_call', id='fc_0f3512ecc7f4f5b10069aeb3a7f784819693dfda887a931d74', status='completed')]
[ResponseFunctionToolCall(arguments='{"city":"Berlin"}', call_id='call_dEQYvPmxM6Fu0uTFFC8QTZK9', name='get_current_weather', type='function_call', id='fc_0f3512ecc7f4f5b10069aeb3abb3c88196ada8a4f653d036ec', status='completed')]
World War II is generally considered to have started in **Germany** with the invasion of Poland on September 1, 1939. 

### Current Weather in Berlin (Germany)
- **Condition:** Broken clouds
- **Temperature:** Approximately 15.7°C (288.84 K)

### Current Weather in Madrid (Spain)
- **Condition:** Overcas